# 06: Volt-VAr slope test: blind AMI vs ground truth vs original SA analysis

Three verdicts per site-month, compared against each other:

| Source | Data | Processing |
|---|---|---|
| **Unmonitored (final method)** | AMI net meter (`ami_meter_residential`) | polarity correction → sign flip + monthly night-time baseline (`base − net`) → **signed net-export gate** (`P_est ≥ 20%` rating) → slope test |
| **Ground truth (direct)** | separate PV circuit (`ami_raw_phaseseparate`) | none: the circuit's own P/Q, the standard's `abs(P) ≥ 20%` gate → same slope test |
| **Original SA** | Athena `conformance_voltvar` (legacy table) | SA's own rule: red non-conformant intervals ≤ 10% |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Put the repo root on sys.path (same as the other notebooks).
_current = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from bms_sa_review.ami_analysis.lib import ami_fleet_vvar as Fleet
from bms_sa_review.ami_analysis.lib import ami_vvar_validation as Valid
from bms_sa_review.ami_analysis.lib import ami_conformance_plot as ConformancePlot

## 1. Settings

In [ ]:
#### ONE MONTH #####
# Scope: start with one month. Add months to extend (e.g. [f"2024-{m:02d}" for m in range(9, 13)])
# MONTHS = ["2024-09"]


#### ALL MONTHS ####
# Scope: every month on disk (2024-01 .. 2025-12), or a hand-picked list such as ["2024-09"]
MONTHS = [p.name.split("=")[1] for p in sorted((Fleet.store_dir() / "ami_meter_residential").glob("dt_month=*"))]
print(f"{len(MONTHS)} months: {MONTHS[0]} .. {MONTHS[-1]}")

# Cohort: None = the WHOLE 1-load/1-PV cohort.
N_SAMPLE_SITES = None
RANDOM_STATE = 42

# Method settings
MAIN_BAND = (242, 253)                       # main scenario: fitted voltage band [v_lo, v_hi)
NIGHT_HOURS = (0, 4)                         # AEST hours for the monthly night-time baseline (blind only)
P_MIN_FRAC = Fleet.P_MIN_FRAC                # capability gate: 20% of rating
MAIN = dict(v_lo=MAIN_BAND[0], v_hi=MAIN_BAND[1], p_min_frac=P_MIN_FRAC, night_hours=NIGHT_HOURS)

# Sensitivity scenarios: only the voltage band changes
RUN_SENSITIVITY = True
SENSITIVITY = {
    "band_240_253": dict(MAIN, v_lo=240, v_hi=253),
#    "band_244_253": dict(MAIN, v_lo=244, v_hi=253),
    "band_240_258": dict(MAIN, v_lo=240, v_hi=258),   # includes the 253-258 V Volt-Watt overlap
}

# Fit quality (same as notebook 07) and verdict rules
MIN_N_WINDOW = 20           # min. 5-min intervals in the fitted window per site-month
MIN_V_SPREAD = 5.0          # min. voltage range (V) those intervals must cover
SLOPE_TOLERANCE_PCT = 20    # slope test: within ±20% of the required slope
SA_RED_THRESHOLD = 0.10     # SA rule: conformant if red intervals <= 10%
POSITIVE_IS_FAIL = False    # metrics' positive class: False = conformant, True = non-conformant
# a POSITIVE_IS_FAIL = False means that in the confusion matrix, the "positive" class is "conformant" and the "negative" class is "non-conformant".

# Outputs (local disk, not OneDrive). One folder per cohort, one sub-folder per scenario
DUCKDB = dict(memory_limit="8GB", threads=4)
COHORT_NAME = "all_sites" if N_SAMPLE_SITES is None else f"sample{N_SAMPLE_SITES}_seed{RANDOM_STATE}"
OUT_DIR = Fleet.store_dir() / "vvar_validation" / COHORT_NAME
OVERWRITE = False           # True re-runs months already on disk (e.g. after changing the cohort)

def scenario_name(s):
    return f"band_{s['v_lo']}_{s['v_hi']}"

print(f"Results folder: {OUT_DIR}")

## 2. Cohort (whole 1-load/1-PV cohort, or the notebook 05 Section 7 sample)

In [ ]:
# Load metadata
con = duckdb.connect()
site_meta = con.sql(f"""
    SELECT * FROM read_parquet('{Fleet.Config.store_path("ami_site_metadata_residential").as_posix()}')
""").df()
con.close()

# 1 load + 1 PV circuit, no flex export, export limit OK
eligibility = Fleet.build_eligibility(site_meta)
eligible_ids = eligibility.loc[eligibility.eligibility == "eligible", "site_id"]

sites = Valid.sample_cohort(site_meta, N_SAMPLE_SITES, RANDOM_STATE)
n_before = len(sites)
sites = sites.loc[sites.site_id.isin(eligible_ids) & (sites.rating_kw > 0)]
print(f"{len(sites):,} sites ({'whole cohort' if N_SAMPLE_SITES is None else 'random sample'}); "
      f"{n_before - len(sites)} removed (flex export, export-limited or no rating)")
display(eligibility.eligibility.value_counts().rename("n_sites").to_frame())

## 3. Example site: blind vs ground truth, interval by interval

- Grey = all intervals
- Blue = the intervals actually fitted (voltage band + capability gate).

In [ ]:
EXAMPLE_SITE_ID = 1084068901
EXAMPLE_MONTH = MONTHS[0]

example_site = (site_meta.loc[site_meta.site_id == EXAMPLE_SITE_ID, ["site_id", "ac_capacity_kw"]]
                .rename(columns={"ac_capacity_kw": "rating_kw"}))

points, fits = {}, {}
for label, source in [("Unmonitored (AMI, final method)", "meter_baseline"), ("Ground truth (direct)", "ground_truth")]:
    points[label] = Fleet.fetch_site_points(EXAMPLE_MONTH, example_site, source, **MAIN)
    # The site's fit for this month, from the same SQL the cohort run uses
    con = Fleet.connect(**DUCKDB)
    con.register("_sites", example_site)
    fit = con.execute(Fleet.month_fit_sql(EXAMPLE_MONTH, source, **MAIN)).df()
    con.close()
    fits[label] = Fleet.add_fit_status(fit, min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD).iloc[0]

Valid.plot_site_month(points, fits, v_lo=MAIN["v_lo"], v_hi=MAIN["v_hi"],
                      title=f"Site {EXAMPLE_SITE_ID}, {EXAMPLE_MONTH}, band {MAIN_BAND[0]}–{MAIN_BAND[1]} V")
plt.show()
display(pd.DataFrame(fits).T[["n_window", "v_min_obs", "v_max_obs", "slope", "r_squared",
                             "base_p_kw", "base_q_kvar", "fit_status"]])

## 4. Main scenario: blind and ground-truth fits for the cohort

In [ ]:
blind, truth = Valid.run_scenario(OUT_DIR / scenario_name(MAIN), MONTHS, sites, MAIN,
                                  min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD,
                                  overwrite=OVERWRITE, **DUCKDB)

# Site-months by fit status, per source
display(pd.concat({"blind": blind.fit_status.value_counts(),
                   "truth": truth.fit_status.value_counts()}, axis=1).fillna(0).astype(int))

### Reference data quality

In [ ]:
truth_all = truth.copy()                    # before exclusion, for the example panel (Section 9)
constant_q = (truth.fit_status == "ok") & (truth.slope == 0) & (truth.slope_stderr == 0)
truth.loc[constant_q, "fit_status"] = "constant_q"
print(f"{constant_q.sum():,} reference site-months set to 'constant_q' "
      f"({truth.loc[constant_q, 'site_id'].nunique():,} sites)")
display(truth.fit_status.value_counts())

In [ ]:
# Slope verdicts (same rule as notebook 05/07: |slope - required| <= tolerance)
blind_pf = ConformancePlot.add_slope_pass_fail(blind.loc[blind.fit_status == "ok"], SLOPE_TOLERANCE_PCT)
truth_pf = ConformancePlot.add_slope_pass_fail(truth.loc[truth.fit_status == "ok"], SLOPE_TOLERANCE_PCT)

# Blind vs ground-truth slopes, where both fits are ok
compare = Fleet.compare_to_truth(blind, truth)
print(f"{len(compare)} site-months with both fits ok; median |slope error| "
      f"{compare.slope_err.abs().median():.3f} %/V")
Fleet.plot_blind_vs_truth(compare)
plt.show()

## 5. Original SA results (Athena)

Queries the **legacy** `conformance_voltvar` table directly (not `TABLES["conformance_voltvar"]`,
which points at the v2 table). Cached to disk, so re-runs don't hit Athena again.

In [ ]:
SA_CACHE = OUT_DIR / f"sa_conformance_voltvar_{MONTHS[0]}_to_{MONTHS[-1]}_{len(MONTHS)}m.parquet"

if SA_CACHE.exists() and not OVERWRITE:
    sa_raw = pd.read_parquet(SA_CACHE)
else:
    from bms_sa_review.shared.aws_config import aq
    from bms_sa_review.shared.ciccada_config import SAI
    sa_raw = aq(Valid.sa_voltvar_sql(sites.site_id, MONTHS), database=SAI)
    SA_CACHE.parent.mkdir(parents=True, exist_ok=True)
    sa_raw.to_parquet(SA_CACHE, index=False)

sa = Valid.add_sa_verdict(sa_raw, SA_RED_THRESHOLD)
sa = sa.loc[sa.site_id.isin(sites.site_id)]                     # same cohort as the fits
print(f"SA: {len(sa)} site-months for {sa.site_id.nunique()} of {len(sites)} cohort sites; "
      f"{sa.passed.notna().sum()} with a verdict")

## 6. Three-way comparison

Each pair answers a different question:

- **Blind vs ground truth** — same test, different data: how well does the net meter stand in for the PV circuit?
- **Ground truth vs SA** — same (PV-circuit) data, different test: how does the slope test compare to SA's rule?
- **Blind vs SA** — both differ: what a reader comparing this method with the original analysis would see.

In [ ]:
combined = Valid.combine_sources(blind_pf, truth_pf, sa)

# Which sources could assess each site-month
display(Valid.coverage_table(combined))

truth_pf_all = ConformancePlot.add_slope_pass_fail(truth_all.loc[truth_all.fit_status == "ok"], SLOPE_TOLERANCE_PCT)
combined_all = Valid.combine_sources(blind_pf, truth_pf_all, sa)   # before the constant-Q exclusion

In [ ]:
# ---------------------------------------------------------------------------
# Cohort figure: (A) response-curve fans, blind vs ground truth; (B) slope verdicts.
# Curves are aggregated in DuckDB (median Q per voltage bin per site-month), so this
# scales to the whole cohort. Only site-months with fit_status == "ok".
# ---------------------------------------------------------------------------
V_BIN = 1.0               # voltage bin width (V)
MIN_BIN_POINTS = 5        # a site-month's bin needs >= this many intervals
MIN_CURVES_PER_BIN = 5    # a bin is drawn only if >= this many site-months contribute

COHORT_SOURCES = {"Unmomitored (AMI, final method)": ("meter_baseline", blind_pf),
                  "Ground truth (direct)": ("ground_truth", truth_pf)}

curves = {}
for label, (source, pf) in COHORT_SOURCES.items():
    frames = []
    for month in MONTHS:
        y, m = int(month[:4]), int(month[5:])
        ok_sites = sites.loc[sites.site_id.isin(pf.loc[(pf.year == y) & (pf.month == m), "site_id"])]
        if not ok_sites.empty:
            frames.append(Fleet.fetch_response_curves(month, ok_sites, source, v_bin=V_BIN,
                                                      min_bin_points=MIN_BIN_POINTS, **DUCKDB, **MAIN))
    curves[label] = pd.concat(frames, ignore_index=True)

fig = Valid.plot_cohort_response_and_verdicts(
    curves, combined, v_lo=MAIN["v_lo"], v_hi=MAIN["v_hi"], tolerance_pct=SLOPE_TOLERANCE_PCT,
    min_curves_per_bin=MIN_CURVES_PER_BIN,
    title=f"{len(sites):,} sites, {', '.join(MONTHS)}, band {MAIN['v_lo']}–{MAIN['v_hi']} V")
fig.savefig(OUT_DIR / scenario_name(MAIN) / "cohort_response_and_verdicts.png", bbox_inches="tight")
plt.show()

# The numbers behind panel A
display(pd.concat({label: Valid.response_fan(cv, MIN_CURVES_PER_BIN) for label, cv in curves.items()},
                  axis=1).round(2))

In [ ]:
# ---------------------------------------------------------------------------
# (B) Slope verdicts, all months pooled: blind slope vs ground-truth slope per site-month,
# with the slope-test pass band shaded. Uses `combined` (Section 6) only -- no new queries.
# ---------------------------------------------------------------------------
req, tol = Fleet.REQUIRED_SLOPE, SLOPE_TOLERANCE_PCT / 100
band = sorted([req * (1 - tol), req * (1 + tol)])          # slopes that pass the test

# Site-months both sources could assess, and whether their verdicts agree
both = combined.dropna(subset=["pass_blind", "pass_truth"]).copy()
both["agree"] = (both.pass_blind == both.pass_truth).astype(bool)

# Axis limits: ignore the most extreme 0.5% of slopes, always show the pass band and 0
slopes = both[["slope_blind", "slope_truth"]]
lim = (min(slopes.quantile(0.005).min(), band[0]) - 0.5, max(slopes.quantile(0.995).max(), 0) + 0.5)
size = 28 if len(both) < 300 else 6                         # smaller markers for large cohorts

fig, ax = plt.subplots(figsize=(7.5, 7.5), dpi=130, layout="constrained")
ax.axvspan(*band, color="#1baf7a", alpha=0.10, linewidth=0)
ax.axhspan(*band, color="#1baf7a", alpha=0.10, linewidth=0, label=f"Pass band (±{SLOPE_TOLERANCE_PCT}%)")
ax.plot(lim, lim, color="black", linewidth=0.8, label="1:1")
ax.scatter(both.loc[both.agree, "slope_truth"], both.loc[both.agree, "slope_blind"], s=size,
           color="#2a78d6", alpha=0.5, edgecolor="white", linewidth=0.2,
           label=f"Same verdict (n={both.agree.sum():,})")
ax.scatter(both.loc[~both.agree, "slope_truth"], both.loc[~both.agree, "slope_blind"], s=size * 2.5,
           marker="X", color="#eb6834", edgecolor="white", linewidth=0.2,
           label=f"Different verdict (n={(~both.agree).sum():,})")

ax.set(xlim=lim, ylim=lim, aspect="equal", xlabel="Ground-truth slope (%/V)", ylabel="Blind slope (%/V)")
ax.set_title(f"Slope verdicts: {both.agree.mean():.1%} agreement (n={len(both):,} site-months)\n"
             f"{len(sites):,} sites, {MONTHS[0]} to {MONTHS[-1]}, band {MAIN['v_lo']}–{MAIN['v_hi']} V",
             fontsize=10, fontweight="bold")
ax.legend(fontsize=8, loc="upper left")
ax.grid(color="#ebebeb", linewidth=0.5)

fig.savefig(OUT_DIR / scenario_name(MAIN) / "slope_verdicts.png", bbox_inches="tight")
plt.show()

# Agreement in numbers, pooled over all months
n_out = int(((slopes < lim[0]) | (slopes > lim[1])).any(axis=1).sum())
print(f"{both.agree.mean():.1%} agreement over {len(both):,} site-months "
      f"({both.site_id.nunique():,} sites); {n_out} site-months fall outside the plotted range")

In [ ]:
# ---------------------------------------------------------------------------
# (B) per month: blind slope vs ground-truth slope for each site-month, one panel per
# month-year, same axes and pass band everywhere. Uses `combined` (Section 6) only.
# ---------------------------------------------------------------------------
PANEL_COLS = 6            # panels per row

req, tol = Fleet.REQUIRED_SLOPE, SLOPE_TOLERANCE_PCT / 100
band = sorted([req * (1 - tol), req * (1 + tol)])          # slopes that pass the test

# Site-months both sources could assess, labelled by month-year and verdict agreement
both = combined.dropna(subset=["pass_blind", "pass_truth"]).copy()
both["agree"] = (both.pass_blind == both.pass_truth).astype(bool)
both["ym"] = both.year.astype(int).astype(str) + "-" + both.month.astype(int).map("{:02d}".format)
months = sorted(both.ym.unique())

# Shared limits: ignore the most extreme 0.5% of slopes, always show the pass band and 0
slopes = both[["slope_blind", "slope_truth"]]
lim = (min(slopes.quantile(0.005).min(), band[0]) - 0.5, max(slopes.quantile(0.995).max(), 0) + 0.5)

ncols = min(PANEL_COLS, len(months))
nrows = -(-len(months) // ncols)                            # ceiling division
fig, axes = plt.subplots(nrows, ncols, figsize=(2.9 * ncols, 2.9 * nrows + 1), dpi=130,
                         sharex=True, sharey=True, squeeze=False, layout="constrained")

for ax, ym in zip(axes.flat, months):
    d = both.loc[both.ym == ym]
    ax.axvspan(*band, color="#1baf7a", alpha=0.12, linewidth=0)
    ax.axhspan(*band, color="#1baf7a", alpha=0.12, linewidth=0)
    ax.plot(lim, lim, color="black", linewidth=0.6)
    ax.scatter(d.loc[d.agree, "slope_truth"], d.loc[d.agree, "slope_blind"],
               s=5, color="#2a78d6", alpha=0.5, linewidth=0)
    ax.scatter(d.loc[~d.agree, "slope_truth"], d.loc[~d.agree, "slope_blind"],
               s=14, marker="X", color="#eb6834", linewidth=0)
    ax.set(xlim=lim, ylim=lim, aspect="equal")
    ax.set_title(f"{ym}: {d.agree.mean():.0%} agree (n={len(d):,})", fontsize=8, fontweight="bold")
    ax.tick_params(labelsize=7)
    ax.grid(color="#ebebeb", linewidth=0.5)

for ax in axes.flat[len(months):]:                          # hide unused grid slots
    ax.set_visible(False)

# One legend and shared axis labels for the whole grid
handles = [
    plt.Line2D([], [], marker="o", linestyle="", color="#2a78d6", label="Same verdict"),
    plt.Line2D([], [], marker="X", linestyle="", color="#eb6834", label="Different verdict"),
    plt.Rectangle((0, 0), 1, 1, color="#1baf7a", alpha=0.3, label=f"Pass band (±{SLOPE_TOLERANCE_PCT}%)"),
    plt.Line2D([], [], color="black", linewidth=0.8, label="1:1"),
]
fig.legend(handles=handles, loc="outside upper center", ncol=4, fontsize=8, frameon=False)
fig.supxlabel("Ground-truth slope (%/V)", fontsize=9)
fig.supylabel("Blind slope (%/V)", fontsize=9)
fig.suptitle(f"Slope verdicts by month: {len(sites):,} sites, {months[0]} to {months[-1]}, "
             f"band {MAIN['v_lo']}–{MAIN['v_hi']} V — overall {both.agree.mean():.1%} agreement "
             f"(n={len(both):,})", fontsize=10, fontweight="bold")
fig.savefig(OUT_DIR / scenario_name(MAIN) / "slope_verdicts_by_month.png", bbox_inches="tight")
plt.show()

# The numbers behind the panels, one row per month
both["abs_slope_err"] = (both.slope_blind - both.slope_truth).abs()
per_month = both.groupby("ym").agg(
    n_site_months=("agree", "size"),
    agreement_pct=("agree", lambda s: 100 * s.mean()),
    pass_blind_pct=("pass_blind", lambda s: 100 * s.astype(bool).mean()),
    pass_truth_pct=("pass_truth", lambda s: 100 * s.astype(bool).mean()),
    median_abs_slope_err=("abs_slope_err", "median"),
)
display(per_month.round(2))
per_month.to_csv(OUT_DIR / scenario_name(MAIN) / "slope_verdicts_by_month.csv")

In [ ]:
positive = "non-conformant" if POSITIVE_IS_FAIL else "conformant"
for common_only in (False, True):
    print(f"\nPositive class = {positive}; "
          + ("site-months ALL three sources assessed" if common_only else "each pair on its own overlap"))
    display(Valid.agreement_table(combined, positive_is_fail=POSITIVE_IS_FAIL, common_only=common_only).round(3))

Valid.plot_confusion_matrices(combined)
plt.show()

In [ ]:
# Disagreements per pair -> CSV (one row per site-month, with each source's numbers)
for tested, reference in Valid.PAIRS:
    d = Valid.disagreements(combined, tested, reference, positive_is_fail=POSITIVE_IS_FAIL)
    path = OUT_DIR / scenario_name(MAIN) / f"disagreements_{tested}_vs_{reference}.csv"
    d.to_csv(path, index=False)
    print(f"{tested} vs {reference}: {(d.outcome == 'false_positive').sum()} FP, "
          f"{(d.outcome == 'false_negative').sum()} FN -> {path.name}")

## 7. Sensitivity: voltage band

Same cohort and rules; only the fitted voltage band changes. One summary row per scenario and pair.

In [ ]:
scenarios = {"main_" + scenario_name(MAIN): MAIN, **(SENSITIVITY if RUN_SENSITIVITY else {})}

rows = []
for name, settings in scenarios.items():
    b, t = Valid.run_scenario(OUT_DIR / scenario_name(settings), MONTHS, sites, settings,
                              min_n_window=MIN_N_WINDOW, min_v_spread=MIN_V_SPREAD,
                              overwrite=OVERWRITE, **DUCKDB)
    t.loc[(t.fit_status == "ok") & (t.slope == 0) & (t.slope_stderr == 0), "fit_status"] = "constant_q"
    b_pf = ConformancePlot.add_slope_pass_fail(b.loc[b.fit_status == "ok"], SLOPE_TOLERANCE_PCT)
    t_pf = ConformancePlot.add_slope_pass_fail(t.loc[t.fit_status == "ok"], SLOPE_TOLERANCE_PCT)
    agree = Valid.agreement_table(Valid.combine_sources(b_pf, t_pf, sa), positive_is_fail=POSITIVE_IS_FAIL)
    rows.append(agree.assign(scenario=name, n_ok_blind=len(b_pf), n_ok_truth=len(t_pf),
                             median_slope_blind=b_pf.slope.median(), median_slope_truth=t_pf.slope.median()))

sensitivity = pd.concat(rows, ignore_index=True)
display(sensitivity.set_index(["scenario", "tested", "reference"])
        [["n", "n_ok_blind", "n_ok_truth", "median_slope_blind", "median_slope_truth",
          "pass_rate_tested_%", "pass_rate_reference_%", "accuracy", "precision", "recall", "f1"]].round(3))

## 8. Save

In [ ]:
# Main-scenario comparison table, its agreement metrics, and the sensitivity summary
main_dir = OUT_DIR / scenario_name(MAIN)
combined.to_csv(main_dir / "three_way_site_month.csv", index=False)
Valid.agreement_table(combined, positive_is_fail=POSITIVE_IS_FAIL).to_csv(main_dir / "agreement.csv", index=False)
sensitivity.to_csv(OUT_DIR / "sensitivity_voltage_band.csv", index=False)
combined_all.to_csv(main_dir / "three_way_site_month_incl_constant_q.csv", index=False)
print(f"Saved to {OUT_DIR}")

# 9. Exploration

In [ ]:
# ---------------------------------------------------------------------------
# The x ~ 0 / blind-pass cluster: ground truth shows no response, blind passes.
# ---------------------------------------------------------------------------
TRUTH_FLAT = 0.5    # |ground-truth slope| below this (%/V) = "no response"

both = combined.dropna(subset=["pass_blind", "pass_truth"]).copy()
cluster = both.loc[(both.slope_truth.abs() < TRUTH_FLAT) & both.pass_blind.astype(bool)]
flat = both.loc[both.slope_truth.abs() < TRUTH_FLAT]

print(f"{len(cluster):,} site-months ({cluster.site_id.nunique():,} sites): ground truth flat, blind passes")
print(f"= {len(cluster) / len(flat):.1%} of the {len(flat):,} flat-ground-truth site-months, "
      f"{len(cluster) / (~both.pass_blind.astype(bool) | ~both.pass_truth.astype(bool)).sum():.1%} "
      f"of all site-months that fail on at least one side")

# 1. Persistence: same sites again and again?
months_per_site = cluster.groupby("site_id").size()
assessed_per_site = both.groupby("site_id").size()
persist = pd.DataFrame({"cluster_months": months_per_site,
                        "assessed_months": assessed_per_site.reindex(months_per_site.index)})
persist["share"] = persist.cluster_months / persist.assessed_months
display(persist.describe().round(2))
print(f"Sites in the cluster in >= 50% of their assessed months: {(persist.share >= 0.5).sum()}")

# 2. Fit quality: clean blind fits suggest a real device, not a coincidence
display(cluster[["slope_blind", "r_squared_blind", "slope_truth", "r_squared_truth"]].describe().round(3))

# 3. Device clues: OEM / DNSP / install year, against the cohort as a whole
attrs = sites[["site_id", "manufacturer", "dnsp_name", "pv_install_date"]].assign(
    install_year=lambda d: d.pv_install_date.dt.year)
cl_sites = attrs.loc[attrs.site_id.isin(cluster.site_id)]
for col in ["manufacturer", "dnsp_name", "install_year"]:
    share = pd.DataFrame({"cluster_%": 100 * cl_sites[col].value_counts(normalize=True),
                          "cohort_%": 100 * attrs[col].value_counts(normalize=True)}).dropna()
    display(share.sort_values("cluster_%", ascending=False).head(8).round(1))

# Save the list for a closer look
cluster.merge(persist, left_on="site_id", right_index=True).to_csv(
    OUT_DIR / scenario_name(MAIN) / "flat_truth_blind_pass_cluster.csv", index=False)

In [ ]:
# ---------------------------------------------------------------------------
# Paper figure: monthly agreement, pass rates and slope error (column width).
# Reads slope_verdicts_by_month.csv saved by the per-month cell -- no new queries.
# ---------------------------------------------------------------------------
import matplotlib.dates as mdates

FIG_DIR = OUT_DIR / scenario_name(MAIN) / "paper_figures"
FIG_DIR.mkdir(exist_ok=True)
COLORS = {"Unmonitored": "#2a78d6", "PV-circuit reference": "#eb6834"}

pm = pd.read_csv(OUT_DIR / scenario_name(MAIN) / "slope_verdicts_by_month.csv", index_col="ym")
x = pd.to_datetime(pm.index)

fig, axes = plt.subplots(3, 1, figsize=(7, 10), dpi=800, sharex=True, layout="constrained",
                         gridspec_kw={"height_ratios": [1, 1.2, 0.8]})

# (a) verdict agreement
axes[0].plot(x, pm.agreement_pct, color="black", marker="o", ms=2.5, lw=1.1)
axes[0].set(ylabel="Agreement (%)", ylim=(85, 100))

# (b) share of site-months called conformant
axes[1].plot(x, pm.pass_blind_pct, color=COLORS["Unmonitored"], marker="o", ms=2.5, lw=1.1, label="Unmonitored")
axes[1].plot(x, pm.pass_truth_pct, color=COLORS["PV-circuit reference"], marker="s", ms=2.5, lw=1.1,
             label="PV-circuit reference")
axes[1].set(ylabel="Conformant (%)", ylim=(10, 30))
axes[1].legend(fontsize=7, frameon=False, loc="lower right")

# (c) median absolute slope error
axes[2].bar(x, pm.median_abs_slope_err, width=20, color="#9ca3af")
axes[2].set(ylabel="Median |error|\n(%/V)", ylim=(0, 0.2))

for ax, letter in zip(axes, "abc"):
    ax.text(0.01, 0.95, f"({letter})", transform=ax.transAxes, fontsize=8, fontweight="bold", va="top")
    ax.grid(color="#ebebeb", linewidth=0.5)
    ax.tick_params(labelsize=7)
    ax.yaxis.label.set_size(8)
axes[2].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

fig.savefig(FIG_DIR / "monthly_agreement.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "monthly_agreement.png", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Paper figure: four example site-months, unmonitored vs PV-circuit reference (2x2).
# (a) both conformant, (b) both no response, (c) false negative, (d) false positive.
# Uses `combined` (constant-Q reference site-months already excluded).
# ---------------------------------------------------------------------------
MIN_N_EXAMPLE = 300       # fitted intervals required in BOTH sources, for clear panels

# Shared settings (also defined in the other paper-figure cells, so this cell runs on its own)
V3 = 240                                    # AS/NZS 4777.2 V3: start of the absorbing ramp
LO, HI = -4.000, -2.667                     # conformance band (%/V), ±20% of required slope
FIG_DIR = OUT_DIR / scenario_name(MAIN) / "paper_figures"
FIG_DIR.mkdir(exist_ok=True)
COLORS = {"Unmonitored": "#2a78d6", "PV-circuit reference": "#eb6834"}


def candidates(frame):
    """Site-months both sources assessed, with enough fitted intervals in both."""
    d = frame.dropna(subset=["pass_blind", "pass_truth"]).copy()
    return d.loc[(d.n_window_blind >= MIN_N_EXAMPLE) & (d.n_window_truth >= MIN_N_EXAMPLE)]


ex = candidates(combined)
pb, pt = ex.pass_blind.astype(bool), ex.pass_truth.astype(bool)


def best(*masks, by):
    """Highest-`by` site-month for the first mask that matches anything."""
    for i, mask in enumerate(masks):
        hit = ex.loc[mask]
        if len(hit):
            if i > 0:
                print(f"  fallback mask {i} used ({len(hit)} candidates)")
            return hit.sort_values(by, ascending=False).iloc[0]
    raise ValueError("No site-month matches any mask; loosen the thresholds.")


def near_req(col):
    """Slope reasonably close to the required -3.333 %/V."""
    return ex[col].between(-3.6, -3.0)


EXAMPLES = {}

print("(a) Both conformant")
EXAMPLES["(a) Both conformant"] = best(
    pb & pt & (ex.site_id == EXAMPLE_SITE_ID) & near_req("slope_truth") & near_req("slope_blind"),
    pb & pt & (ex.site_id == EXAMPLE_SITE_ID),                     # same site, any pass/pass month
    pb & pt & near_req("slope_truth") & near_req("slope_blind"),   # any site
    by="n_window_truth")

print("(b) Both non-conformant")
EXAMPLES["(b) Both non-conformant"] = best(
    ~pb & ~pt & (ex.slope_truth.abs() < 0.3) & (ex.slope_blind.abs() < 0.3),
    by="n_window_truth")

print("(c) False negative")
EXAMPLES["(c) False negative"] = best(
    ~pb & pt & near_req("slope_truth") & (ex.slope_blind > -2.4),  # clearly flattened
    ~pb & pt & near_req("slope_truth") & (ex.slope_blind > -2.5),
    by="r_squared_truth")

print("(d) False positive")
EXAMPLES["(d) False positive"] = best(
    pb & ~pt & near_req("slope_blind") & (ex.slope_truth > -2.2),  # clearly outside the band
    pb & ~pt & (ex.slope_truth > -2.4),
    pb & ~pt,
    by="r_squared_blind")

# Sanity check: stored verdicts must match the band
for title, r in EXAMPLES.items():
    for s, p in [("blind", "pass_blind"), ("truth", "pass_truth")]:
        in_band = LO <= r[f"slope_{s}"] <= HI
        assert bool(r[p]) == in_band, f"{title}: {s} slope {r[f'slope_{s}']:.3f} vs pass={r[p]}"

rating = site_meta.set_index("site_id").ac_capacity_kw
SOURCES_PLOT = [("meter_baseline", "Unmonitored"), ("ground_truth", "PV-circuit reference")]
v = np.linspace(MAIN["v_lo"], MAIN["v_hi"], 50)


def plot_example(ax, title, r, rows):
    """Fitted-window points and OLS line for both sources of one site-month, on one axis."""
    month = f"{int(r.year)}-{int(r.month):02d}"
    site = pd.DataFrame({"site_id": [r.site_id], "rating_kw": [rating[r.site_id]]})
    for source, name in SOURCES_PLOT:
        win = Fleet.fetch_site_points(month, site, source, **MAIN).query("in_window")
        slope, icpt = np.polyfit(win.V, win.q_pct, 1)          # same OLS as the SQL fit
        ax.scatter(win.V, win.q_pct, s=2, alpha=0.25, color=COLORS[name], linewidth=0, rasterized=True)
        ax.plot(v, slope * v + icpt, color=COLORS[name], lw=1.6, label=f"{name}: {slope:.2f} %/V")
        tag = "blind" if source == "meter_baseline" else "truth"
        rows.append(dict(panel=title, site_id=r.site_id, month=month, source=name, n=len(win),
                         slope_refit=slope, slope_stored=r[f"slope_{tag}"], passed=bool(r[f"pass_{tag}"])))
    ax.plot(v, Fleet.REQUIRED_SLOPE * (v - V3), color="black", linestyle="--", lw=1.1, label="Required")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"{title}, {month}", fontsize=8, fontweight="bold")
    ax.legend(fontsize=6.5, frameon=False, loc="lower left")
    ax.grid(color="#ebebeb", linewidth=0.5)
    ax.tick_params(labelsize=7)
    ax.set(xlim=(MAIN["v_lo"], MAIN["v_hi"]), ylim=(-55, 25))


fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.0), dpi=300, sharex=True, sharey=True, layout="constrained")
rows = []
for ax, (title, r) in zip(axes.flat, EXAMPLES.items()):
    plot_example(ax, title, r, rows)

fig.supxlabel("Voltage (V)", fontsize=8)
fig.supylabel("Reactive power (% of rating)  + supplying, − absorbing", fontsize=8)
fig.savefig(FIG_DIR / "example_sites.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "example_sites.png", bbox_inches="tight")
plt.show()

display(pd.DataFrame(rows).round(3))      # slopes for the text; refit should match stored

In [ ]:
# ---------------------------------------------------------------------------
# Separate figure: a site-month EXCLUDED from validation because the PV circuit
# reports constant reactive power. Uses `combined_all` (before the exclusion).
# Reuses candidates(), plot_example(), rating, SOURCES_PLOT and v from the cell above.
# ---------------------------------------------------------------------------
ex_all = candidates(combined_all)
excluded = ex_all.loc[ex_all.pass_blind.astype(bool) & (ex_all.slope_truth == 0)]
print(f"{len(excluded):,} candidate constant-Q site-months (unmonitored passes, ≥{MIN_N_EXAMPLE} intervals each)")
r_cq = excluded.sort_values("r_squared_blind", ascending=False).iloc[0]

fig, ax = plt.subplots(figsize=(3.6, 3.2), dpi=300, layout="constrained")
rows_cq = []
plot_example(ax, "Constant reference $Q$ (excluded from validation)", r_cq, rows_cq)
ax.set_xlabel("Voltage (V)", fontsize=8)
ax.set_ylabel("Reactive power (% of rating)\n+ supplying, − absorbing", fontsize=8)
fig.savefig(FIG_DIR / "example_constant_q_excluded.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "example_constant_q_excluded.png", bbox_inches="tight")
plt.show()

display(pd.DataFrame(rows_cq).round(3))